# 02 Refh Quality and DSM

Purpose:
- inspect `refh_snr`, `refh_amp`, `good_snr`, and refh height patterns,
- compare multiple SNR thresholds for point count and spatial coverage,
- view a strict refh-based DSM result.

Scientific notes:
- The DSM shown here is a max-Rx-bin/refh surface product.
- It is not a classified ground DEM.
- Empty cells should remain NoData in the strict workflow unless you intentionally change that policy.

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
import rasterio

H5_PATH = Path('../casals_h5_downloads/casals_l1b_20241112T165718_001_02.h5')
DSM_PATH = Path('../dsm/casals_l1b_20241112T165718_001_02_refh_surface_dsm_snr5_10m_epsg32618.tif')

In [ ]:
with h5py.File(H5_PATH, 'r') as h5:
    lon = h5['refh_longitude'][:]
    lat = h5['refh_latitude'][:]
    refh = h5['refh'][:]
    snr = h5['refh_snr'][:]
    amp = h5['refh_amp'][:]
    good = h5['good_snr'][:].astype(bool)

base = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(refh) & np.isfinite(snr) & np.isfinite(amp)
thresholds = [2.0, 3.0, 4.0, 5.0]
for thr in thresholds:
    keep = base & (snr >= thr)
    print({'snr_threshold': thr, 'n_points': int(np.sum(keep)), 'fraction': float(np.mean(keep))})
print({'good_snr_fraction': float(np.mean(good[base]))})

In [ ]:
sample = np.random.default_rng(42).choice(np.flatnonzero(base), size=min(200_000, int(np.sum(base))), replace=False)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].scatter(lon[sample], lat[sample], c=snr[sample], s=0.2, cmap='viridis', linewidths=0)
axes[0, 0].set_title('Footprint by refh_snr')
axes[0, 1].scatter(lon[sample], lat[sample], c=amp[sample], s=0.2, cmap='magma', linewidths=0)
axes[0, 1].set_title('Footprint by refh_amp')
axes[1, 0].scatter(lon[sample], lat[sample], c=refh[sample], s=0.2, cmap='terrain', linewidths=0)
axes[1, 0].set_title('Footprint by refh height')
axes[1, 1].hist(snr[base], bins=120, color='0.35')
axes[1, 1].axvline(5.0, color='r', linestyle='--', label='good_snr reference')
axes[1, 1].legend()
axes[1, 1].set_title('refh_snr distribution')
fig.tight_layout()

In [ ]:
with rasterio.open(DSM_PATH) as src:
    dsm = src.read(1)
    nodata = src.nodata
valid = np.isfinite(dsm) if nodata is None else np.isfinite(dsm) & (dsm != nodata)
print({'shape': dsm.shape, 'valid_cells': int(np.sum(valid)), 'valid_fraction': float(np.mean(valid))})
plt.figure(figsize=(9, 8))
plt.imshow(np.where(valid, dsm, np.nan), cmap='terrain')
plt.colorbar(label='refh surface height (m)')
plt.title('Strict refh-based DSM')
plt.tight_layout()